## 0. Mount Google Drive và cài thư viện

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install nnAudio einops timm
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Runtime M2D (mã `portable_m2d.py` của repo nttcslab/m2d, đặt trực tiếp trong notebook)

Ô này định nghĩa lớp `PortableM2D` dùng để nạp checkpoint M2D-CLAP, tính log-mel và encode audio/text.

In [ ]:
"""Masked Modeling Duo (M2D) Portable Runtime.

Basic requirements:
    pip install timm einops nnAudio transformers

Additional requirements for CLAP:
    pip install sentence_transformers
"""

import logging
import os
import numpy as np
from pathlib import Path
from functools import partial
import re

import torch
import timm
from timm.layers import trunc_normal_
from einops import rearrange
import nnAudio.features


class Config:
    weight_file = ''
    feature_d = 768 * 5
    model = ''
    input_size = [80, 208]
    patch_size = [16, 16]
    sr = '16k'
    flat_features = False


def expand_size(sz):
    if isinstance(sz, int):
        return [sz, sz]
    return sz


class PatchEmbed(torch.nn.Module):
    """ 2D Image to Patch Embedding -- borrowed from https://pypi.org/project/timm/0.4.12/"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768, norm_layer=None, flatten=True):
        super().__init__()
        img_size = expand_size(img_size)
        patch_size = expand_size(patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten

        self.proj = torch.nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim) if norm_layer else torch.nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)  # BCHW -> BNC
        x = self.norm(x)
        return x


class LocalViT(timm.models.vision_transformer.VisionTransformer):
    """ Vision Transformer for M2D Audio"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Workaround for PatchEmbed to avoid unintended assertion failure. ex) AssertionError: Input image width (102) doesn't match model (608).
        self.patch_embed = PatchEmbed(self.patch_embed.img_size, self.patch_embed.patch_size,
                                      self.patch_embed.proj.in_channels, self.patch_embed.proj.out_channels)
        self.norm_stats = torch.nn.Parameter(torch.tensor([-7.1, 4.2]), requires_grad=False)
        # We do not use the default head
        del self.head

    def patch_size(self):
        return np.array(self.patch_embed.patch_size)

    def grid_size(self):
        # Workaround for compatibility issue (timm 0.4.5 fails with: return self.patch_embed.grid_size)
        img_size = np.array(self.patch_embed.img_size)
        patch_size = self.patch_size()
        grid_size = img_size // patch_size
        return grid_size

    def forward_encoder(self, x):
        x = self.patch_embed(x)

        # add pos embed w/o cls token
        pos_embed = self.pos_embed[:, 1:, :]
        if x.shape[1] < pos_embed.shape[1]:  # shorten pos_embed for a short input
            dims = pos_embed.shape[-1]
            fbins = self.grid_size()[0]
            frames = x.shape[1] // fbins
            pos_embed = pos_embed.reshape(1, fbins, -1, dims)[:, :, :frames, :].reshape(1, fbins*frames, dims)
        x = x + pos_embed

        # append cls token
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # apply Transformer blocks
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)

        return x


class AudioSemanticProj(torch.nn.Module):
    def __init__(self, embed_dim=768, sem_depth=1, sem_heads=1, sem_mlp_ratio=1):
        super().__init__()
        self.sem_token = torch.nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.sem_blocks = torch.nn.ModuleList([
            timm.models.vision_transformer.Block(embed_dim, sem_heads, sem_mlp_ratio, qkv_bias=True, norm_layer=torch.nn.LayerNorm)
            for i in range(sem_depth)])
        self.norm = torch.nn.LayerNorm(embed_dim)
        self.apply(self._init_weights)
        self.dont_average = True

    def _init_weights(self, m):
        if isinstance(m, torch.nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if isinstance(m, torch.nn.Linear) and m.bias is not None:
                torch.nn.init.constant_(m.bias, 0)
        elif isinstance(m, torch.nn.LayerNorm):
            torch.nn.init.constant_(m.bias, 0)
            torch.nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        # append sem token
        sem_token = self.sem_token  # + self.pos_embed[:, :1, :]
        sem_tokens = sem_token.expand(x.shape[0], -1, -1)
        x = torch.cat((sem_tokens, x), dim=1)

        # apply Transformer blocks
        for blk in self.sem_blocks:
            x = blk(x)
        x = x[:, 0, :]  # use semantic token only
        x = self.norm(x)

        return x


def set_requires_grad(model, val):
    for p in model.parameters():
        p.requires_grad = val


def parse_sizes_by_name(name):
    # Parse parameters. "m2d_vit_base-80x1001p16x16p16kpXXXpYYY" -> input size: 80x1001, patch size: 16x16, sr: 16k, extra parameters: XXX and YYY
    model_cls = name.split('-')[0]
    params = name.split('-')[1]
    params = params.split('p')
    params, extra = params[:3], params[3:]
    input_str, patch_str, sr = params[0], params[1], params[2] if len(params) > 2 else '16k'
    input_size = [int(a) for a in input_str.split('x')]
    patch_size = [int(a) for a in patch_str.split('x')]
    return input_size, patch_size, sr, model_cls, extra


def drop_non_model_weights(model, checkpoint, filename, except_for=[]):  # except_for=['running_mean', 'running_var']
    def not_an_exception(k):
        for ex in except_for:
            if ex in k: return False
        return True
    model_keys = [n for n, p in model.named_parameters()]
    new_ckpt = {}
    dropped = []
    for k in checkpoint:
        if k not in model_keys and not_an_exception(k):
            dropped.append(k)
            continue
        new_ckpt[k] = checkpoint[k]
    n_org = len(checkpoint.keys())
    n_cur = len(new_ckpt.keys())
    print(f' using {n_cur} parameters, while dropped {n_org - n_cur} out of {n_org} parameters from {Path(filename).parent/Path(filename).name}'
          if n_org > n_cur else f' using {n_cur} parameters from {Path(filename).parent/Path(filename).name}')
    print(' (included audio_proj params:', [k for k in new_ckpt.keys() if 'audio_proj' in k][:5])
    print(' (included text_proj params:', [k for k in new_ckpt.keys() if 'text_proj' in k][:5])
    print(' (dropped:', dropped[:5], ')' if len(dropped) < 5 else '...)')
    return new_ckpt


def load_evar_head_parameters(checkpoint, head_norm, head):
    # Load the weights of the task head trained in the EVAR fine-tuning.
    if 'module.head.norm.running_mean' in checkpoint:
        head_norm.load_state_dict({to_k: checkpoint[k] for to_k, k in {
            'running_mean':'module.head.norm.running_mean', 'running_var':'module.head.norm.running_var'}.items()})
        head.load_state_dict({to_k: checkpoint[k] for to_k, k in {
            'weight':'module.head.mlp.mlp.0.weight', 'bias':'module.head.mlp.mlp.0.bias'}.items()})
    else:
        print(' Not an EVAR checkpoint for loading head weights.')


def reformat_ckpt_keys(checkpoint):
    # In case: checkpoint['model']
    checkpoint = checkpoint['model'] if 'model' in checkpoint else checkpoint
    # The checkpoints saved in a EVAR fine-tuning has a prefix of "module.ar.runtime.backbone", the following removes it.
    new_ckpt = {}
    for k in checkpoint:
        new_k = k.replace('module.ar.runtime.backbone.', '')  # replace
        new_ckpt[new_k] = checkpoint[k]
    return new_ckpt


def extract_weight(checkpoint, root_name):
    # if no key starts with root_name, not applicable to the checkpoint.
    if not any(k.startswith(root_name) for k in checkpoint.keys()):
        return checkpoint
    # keep only the items starts with the root_name
    new_ckpt = {k[len(root_name):]: v for k, v in checkpoint.items() if k.startswith(root_name)}
    return new_ckpt


def add_semantic_audio_proj(sem_mode, embed_dim):
    sem_params = {
        1: {'sem_depth': 1, 'sem_heads': 1, 'sem_mlp_ratio': 1},
        2: {'sem_depth': 2, 'sem_heads': 1, 'sem_mlp_ratio': 1},
    }[sem_mode]
    audio_proj = AudioSemanticProj(embed_dim=embed_dim, **sem_params)
    return audio_proj


def make_it_CLAP(model, checkpoint):
    # Return if already a CLAP model
    if hasattr(model, 'audio_proj') or checkpoint is None: return
    # Add projectors if needed
    if 'audio_proj.0.weight' in checkpoint.keys():
        print(checkpoint['audio_proj.0.weight'].shape)
        proj_hidden_dim, embed_dim = checkpoint['audio_proj.0.weight'].shape
        model.audio_proj = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, proj_hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(proj_hidden_dim, proj_hidden_dim),
        )
    if 'audio_proj.sem_token' in checkpoint.keys():
        embed_dim = checkpoint['audio_proj.sem_token'].shape[-1]
        sem_blocks_nums = [int(k.split('.')[2]) for k in checkpoint.keys() if k.startswith('audio_proj.sem_blocks.')]
        sem_mode = max(sem_blocks_nums) + 1
        model.audio_proj = add_semantic_audio_proj(sem_mode, embed_dim)
    if 'text_proj.weight' in checkpoint.keys():
        dim = checkpoint['text_proj.weight'].shape
        model.text_proj = torch.nn.Linear(dim[1], dim[0])


def parse_clap_type(name):
    # Parse parameters. "m2d_clap_base-80x1001p16x16p16kpA" -> input size: 80x1001, patch size: 16x16, sr: 16k, extra: A
    params = str(name).split('-')[1]
    params = params.split('p')
    params, extra = params[:3], params[3:]
    if len(extra) == 0:
        return 'A'
    assert extra[0] in 'ABCLMNQR'
    text_encoder_name = {'A': 'GTE base', 'B': 'BERT base', 'C': 'CLAP-L', 'L': 'GTE large', 'M': 'BERT large', 'N': 'NV-Embed-v2',
                         'Q': 'RoBERTa large', 'R': 'RoBERTa base'}
    logging.info(f' using text encoder: {text_encoder_name[extra[0]]}')
    return extra[0]


def clap_text_enc_dim(param_extra):
    if len(param_extra) == 0:
        return 768
    return {'A': 768, 'B': 768, 'C': 1024, 'L': 1024, 'M': 1024, 'N': 4096, 'Q': 1024, 'R': 768}[param_extra[0]]


def get_backbone(args, weight_file):
    args.input_size, args.patch_size, args.sr, args.model, extra = parse_sizes_by_name(Path(weight_file).parent.name)

    # Load checkpoint.
    checkpoint = torch.load(weight_file, map_location='cpu', weights_only=False)
    checkpoint = checkpoint['model'] if 'model' in checkpoint else checkpoint
    checkpoint = extract_weight(checkpoint, 'backbone.') if checkpoint is not None else None  # convert from RuntimeM2D weights
    checkpoint = reformat_ckpt_keys(checkpoint)
    # Set normalization statistics for backward compatibility. The [-7.1, 4.2] is for 2022 models.
    if 'norm_stats' not in checkpoint:
        checkpoint['norm_stats'] = torch.tensor([-7.1, 4.2])
        print(' using default norm_stats:', checkpoint['norm_stats'])

    # Create a ViT.
    model = LocalViT(
        in_chans=1, img_size=args.input_size, patch_size=args.patch_size, embed_dim=768, depth=12, num_heads=12,
        mlp_ratio=4, norm_layer=partial(torch.nn.LayerNorm, eps=1e-6))

    # Modify the model if it should be a M2D-CLAP.
    make_it_CLAP(model, checkpoint)
    if len(extra) > 1 and extra[1] == 'TI':  # text encoder included
        args.text_encoder_weight = weight_file

    # Load weights.
    dropped = drop_non_model_weights(model, checkpoint, weight_file)
    msg = model.load_state_dict(dropped)
    print(msg); logging.info(msg)

    # Make normalization statistics for the model easy to use in the downstream task.
    args.mean, args.std = model.state_dict()['norm_stats'].to('cpu').numpy()

    model.eval()
    return model, checkpoint


def get_to_melspec(cfg):
    if cfg.sr == '16k':
        cfg.sample_rate, cfg.n_fft, cfg.window_size, cfg.hop_size = 16000, 400, 400, 160
        cfg.n_mels, cfg.f_min, cfg.f_max = 80, 50, 8000
    elif cfg.sr == '32k':
        cfg.sample_rate, cfg.n_fft, cfg.window_size, cfg.hop_size = 32000, 800, 800, 320
        cfg.n_mels, cfg.f_min, cfg.f_max = 80, 50, 16000
    else:
        assert False, f'Unknown input size: {cfg.input_size}'

    to_spec = nnAudio.features.MelSpectrogram(
        sr=cfg.sample_rate,
        n_fft=cfg.n_fft,
        win_length=cfg.window_size,
        hop_length=cfg.hop_size,
        n_mels=cfg.n_mels,
        fmin=cfg.f_min,
        fmax=cfg.f_max,
        center=True,
        power=2,
        verbose=False,
    )
    logging.info(f'Runtime MelSpectrogram({cfg.sample_rate}, {cfg.n_fft}, {cfg.window_size}, {cfg.hop_size}, '
                 + f'{cfg.n_mels}, {cfg.f_min}, {cfg.f_max}):')
    logging.info(to_spec)
    return to_spec


def get_timestamps(cfg, batch_audio, x):  # Returns timestamps in milliseconds.
    audio_len = len(batch_audio[0])
    sec = audio_len / cfg.sample_rate
    x_len = len(x[0])
    step = sec / x_len * 1000 # sec -> ms
    ts = torch.tensor([step * i for i in range(x_len)]).unsqueeze(0)
    ts = ts.repeat(len(batch_audio), 1)
    return ts


class PortableM2D(torch.nn.Module):
    def __init__(self, weight_file, cfg=Config(), num_classes=None, freeze_embed=False, flat_features=None):
        super().__init__()
        self.cfg = cfg
        self.cfg.weight_file = weight_file
        self.cfg.freeze_embed = freeze_embed
        self.cfg.flat_features = self.cfg.flat_features if flat_features is None else flat_features

        # Create backbone model.
        self.backbone, checkpoint = get_backbone(self.cfg, self.cfg.weight_file)
        # Finalize feature dimension.
        d = self.backbone.pos_embed.shape[-1]
        if num_classes is not None and 'module.head.mlp.mlp.0.weight' in checkpoint and checkpoint['module.head.mlp.mlp.0.weight'].shape[-1] == d:
            self.cfg.flat_features = True
        n_stack_feature = 1 if self.cfg.flat_features else (self.cfg.input_size[0] // self.cfg.patch_size[0])
        self.cfg.feature_d = d * n_stack_feature  # 768 if flat_features else 768*5=3840
        # Create head.
        if num_classes is not None:
            self.head_norm = torch.nn.BatchNorm1d(self.cfg.feature_d, affine=False)
            self.head = torch.nn.Linear(self.cfg.feature_d, num_classes)
            trunc_normal_(self.head.weight, std=2e-5)
            load_evar_head_parameters(checkpoint, self.head_norm, self.head)
        # Option: freeze patch embedding ([2211.09359] How to Fine-Tune Vision Models with SGD)
        if self.cfg.freeze_embed:
            set_requires_grad(self.backbone.patch_embed, False)
            logging.info(' ** Freeze patch_embed **')
            logging.info(self.backbone.patch_embed)

        logging.info(f'Model input size: {self.cfg.input_size}')
        logging.info(f'Using weights: {self.cfg.weight_file}')
        logging.info(f'Feature dimension: {self.cfg.feature_d}')
        logging.info(f'Norm stats: {self.cfg.mean}, {self.cfg.std}')

        self.to_spec = get_to_melspec(self.cfg)
        self.eval()

    def to_log_mel_spec(self, batch_audio):
        x = self.to_spec(batch_audio)
        x = (x + torch.finfo().eps).log()
        x = x.unsqueeze(1)
        return x

    def normalize_batch(self, x):
        x = (x - self.cfg.mean) / self.cfg.std
        return x

    def to_normalized_feature(self, batch_audio):
        x = self.to_log_mel_spec(batch_audio)
        x = self.normalize_batch(x)
        return x

    def encode_lms(self, x, average_per_time_frame=False):
        patch_fbins = self.backbone.grid_size()[0]
        unit_frames = self.cfg.input_size[1]
        patch_frames = self.backbone.patch_size()[1]
        embed_d = self.backbone.patch_embed.proj.out_channels
        n_chunk = (x.shape[-1] + unit_frames - 1) // unit_frames
        pad_frames = (patch_frames - (x.shape[-1] % unit_frames % patch_frames)) % patch_frames
        if pad_frames > 0:
            x = torch.nn.functional.pad(x, (0, pad_frames))

        embeddings = []
        if self.cfg.flat_features:
            # flatten all patch embeddings
            for i in range(n_chunk):
                emb = self.backbone.forward_encoder(x[..., i*unit_frames:(i+1)*unit_frames])
                emb = emb[..., 1:, :]
                if average_per_time_frame:
                    emb = rearrange(emb, 'b (f t) d -> b t d f', f=patch_fbins, d=embed_d).mean(-1)
                embeddings.append(emb)
        else:
            # stack embeddings along time frame
            for i in range(n_chunk):
                emb = self.backbone.forward_encoder(x[..., i*unit_frames:(i+1)*unit_frames])
                emb = emb[..., 1:, :]
                emb = rearrange(emb, 'b (f t) d -> b t (f d)', f=patch_fbins, d=embed_d)
                embeddings.append(emb)
        # concatenate embedding chunks in the time axis
        x = torch.cat(embeddings, axis=-2)
        return x

    def encode(self, batch_audio, average_per_time_frame=False):
        x = self.to_normalized_feature(batch_audio)
        return self.encode_lms(x, average_per_time_frame=average_per_time_frame)

    def forward(self, batch_audio, average_per_time_frame=False):
        x = self.encode(batch_audio, average_per_time_frame=average_per_time_frame)
        if hasattr(self, 'head'):
            x = x.mean(1)  # B, D
            x = self.head_norm(x.unsqueeze(-1)).squeeze(-1)
            x = self.head(x)
        return x

    def get_scene_embeddings(self, batch_audio):
        x = self.encode(batch_audio)
        x = torch.mean(x, dim=1)
        return x

    def get_timestamp_embeddings(self, batch_audio):
        x = self.encode(batch_audio, average_per_time_frame=True)
        ts = get_timestamps(self.cfg, batch_audio, x)
        return x, ts

    def forward_frames(self, batch_audio):
        x, ts = self.get_timestamp_embeddings(batch_audio)
        if hasattr(self, 'head'):
            x = self.head_norm(x.transpose(-1, -2)).transpose(-2, -1)
            x = self.head(x)
        return x, ts

    def encode_clap_audio(self, batch_audio):
        audio_embeddings = self.forward(batch_audio)
        if not hasattr(self.backbone.audio_proj, 'dont_average'):
            audio_embeddings = audio_embeddings.mean(dim=-2)
        audio_embeddings = self.backbone.audio_proj(audio_embeddings)
        return audio_embeddings

    def encode_clap_text(self, batch_text, truncate=False):
        if not hasattr(self, 'text_encoder'):
            self.get_clap_text_encoder()
        text_embeddings = self.text_encoder(batch_text, truncate=truncate)
        if hasattr(self.backbone, 'text_proj'):
            text_embeddings = self.backbone.text_proj(text_embeddings)
        return text_embeddings

    def get_clap_text_encoder(self):
        text_encoder_weight = self.cfg.text_encoder_weight if hasattr(self.cfg, 'text_encoder_weight') else None
        self.text_encoder = get_text_encoder(self.cfg.weight_file, text_encoder_weight=text_encoder_weight)
        self.text_encoder = self.text_encoder.to(next(self.backbone.parameters()).device)


# For the CLAP models

class GTETextEncoder(torch.nn.Module):
    def __init__(self, clip_weight="thenlper/gte-base"):
        super().__init__()
        from transformers import AutoTokenizer, AutoModel
        import os
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        cache_dir = os.environ.get("HF_HOME")
        self.tokenizer = AutoTokenizer.from_pretrained(clip_weight, cache_dir=cache_dir)
        self.model = AutoModel.from_pretrained(clip_weight, cache_dir=cache_dir)

    def __call__(self, texts, truncate=True, max_length=512):
        def average_pool(last_hidden_states, attention_mask):
            last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
            return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

        device = next(self.model.parameters()).device
        batch_dict = self.tokenizer(texts, max_length=max_length, padding=True, truncation=truncate, return_tensors='pt')
        batch_dict['input_ids'] = batch_dict['input_ids'].to(device)
        batch_dict['token_type_ids'] = batch_dict['token_type_ids'].to(device)
        batch_dict['attention_mask'] = batch_dict['attention_mask'].to(device)
        outputs = self.model.to(device)(**batch_dict)
        embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
        return embeddings


class GTEL15Encoder(torch.nn.Module):
    def __init__(self, clip_weight="Alibaba-NLP/gte-large-en-v1.5"):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        import os
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        self.model = SentenceTransformer(clip_weight, trust_remote_code=True)

    def __call__(self, texts, **kwargs):
        embeddings = self.model.encode(texts, show_progress_bar=False, convert_to_tensor=True)
        return embeddings


class NVEmbedV2Encoder(torch.nn.Module):
    def __init__(self, clip_weight="nvidia/NV-Embed-v2"):
        # https://huggingface.co/spaces/mteb/leaderboard https://huggingface.co/nvidia/NV-Embed-v2
        # https://arxiv.org/pdf/2405.17428
        from transformers import AutoModel
        super().__init__()
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        self.model = AutoModel.from_pretrained(
            clip_weight, trust_remote_code=True, cache_dir=os.environ.get("HF_HOME")
        )

    def __call__(self, texts, **kwargs):
        embeddings = self.model.encode(texts, instruction="", max_length=32768)
        return embeddings


class CLIPLTextEncoder(torch.nn.Module):
    def __init__(self, clip_weight="ViT-L/14"):
        super().__init__()
        import clip

        device = 'cpu'  # "cuda" if torch.cuda.is_available() else "cpu"
        self.clip = clip
        self.model, _ = clip.load(clip_weight, device=device)

    def __call__(self, texts, truncate=True, max_length=77):
        device = next(self.model.parameters()).device
        tokens = self.clip.tokenize(texts, context_length=max_length, truncate=True).to(device)
        embeddings = self.model.to(device).encode_text(tokens)
        return embeddings


class BertXEncoder(torch.nn.Module):
    def __init__(self, clip_weight="google-bert/bert-base-uncased"):
        super().__init__()
        from transformers import AutoTokenizer, AutoModel
        import os
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        cache_dir = os.environ.get("HF_HOME")
        self.tokenizer = AutoTokenizer.from_pretrained(clip_weight, cache_dir=cache_dir)
        self.text_encoder = AutoModel.from_pretrained(clip_weight, cache_dir=cache_dir)

    def forward(self, batch_text, truncate=True, max_length=512):
        device = next(self.text_encoder.parameters()).device
        text_input = self.tokenizer(batch_text,
            padding='longest',
            truncation=truncate,
            max_length=max_length,
            return_tensors="pt").to(device)
        text_feats = self.text_encoder(input_ids=text_input.input_ids,
            attention_mask=text_input.attention_mask)[0]
        text_feats = text_feats[:, 0, :]
        return text_feats


def get_text_encoder(weight, text_encoder_weight=None):
    try:
        clap_type = parse_clap_type(Path(weight).parent.name)
    except:
        clap_type = parse_clap_type(Path(weight).stem)

    if clap_type == 'A':
        text_model = GTETextEncoder()
    if clap_type == 'B':
        text_model = BertXEncoder()
    if clap_type == 'C':
        text_model = CLIPLTextEncoder()
    if clap_type == 'L':
        text_model = GTEL15Encoder()
    if clap_type == 'M':
        text_model = BertXEncoder(clip_weight="google-bert/bert-large-uncased")
    if clap_type == 'N':
        text_model = NVEmbedV2Encoder()
    if clap_type == 'Q':
        text_model = BertXEncoder(clip_weight="FacebookAI/roberta-large")
    if clap_type == 'R':
        text_model = BertXEncoder(clip_weight="FacebookAI/roberta-base")

    if text_encoder_weight is not None:
        weights = torch.load(text_encoder_weight, map_location='cpu', weights_only=False)
        weights = weights['model'] if 'model' in weights else weights
        if any(['module.ar.runtime.' in k for k in weights]):
            print(f' loading EVAR weight {text_encoder_weight} by removing "module.ar.runtime." from keys.')
            renamed = {k.replace('module.ar.runtime.', ''): weights[k] for k in weights}
            weights = renamed
        weights = extract_weight(weights, 'text_encoder.')
        print(f' using model.text_encoder from {text_encoder_weight}')
        text_model.load_state_dict(weights)
    return text_model

## 2. Cấu hình

In [ ]:
import os, sys, math, time, json, glob, shutil, zipfile, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr, pearsonr, kendalltau

# ---- đường dẫn trên Drive (để "" thì tự tìm) -------------------------------------------
DATA_FOLDER_ID   = "1AV1YTcjgrtuBv_-T07OSTrak1urdAysQ"   # XACLE_Challenge_2027_development_dataset
PARENT_FOLDER_ID = "1Klm-oDzQAhdbabamKUGH-FjUHdZNXkkg"   # thư mục cha "XACLE"
DATA_NAME  = "XACLE_Challenge_2027_development_dataset"
DRIVE_DATA = ""          # để trống = tự tìm theo ID; hoặc ghi thẳng đường dẫn đã mount
COPY_LOCAL = True        # True: chép về /content trước (nhanh hơn nhiều khi train); False: đọc thẳng từ Drive
DRIVE_CKPT = ""          # vd "/content/drive/MyDrive/xacle2027/checkpoint-30.pth" ("" = tự tìm, không có thì tải)
DRIVE_OUT  = "/content/drive/MyDrive/xacle2027/e31_lora_outputs"   # nơi lưu kết quả

CFG = dict(
    variant="audio_text",   # "audio" = LoRA chỉ nhánh audio (A); "audio_text" = thêm BERT (B)
    seed=7,                 # MỘT seed (Rule 7 cấm trung bình nhiều model)
    epochs=5, patience=2,
    batch_size=16, eval_batch=64,
    lr_head=1e-3, lr_lora=5e-4, weight_decay=1e-2,
    lora_r=8, lora_alpha=16,
    lora_blocks=4,          # số block cuối ViT audio gắn LoRA
    lora_text_layers=2,     # số layer cuối BERT gắn LoRA (chỉ với "audio_text")
    workers=2,              # Colab có 2 nhân CPU
    run_logo=False,         # True: LOGO 9 fold (~45–60 phút trên T4)
    smoke=False,            # True: chạy thử 64 mẫu, 1 epoch để kiểm tra đường dẫn
)
SR, DUR, N_LEVELS = 16000, 10, 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path(DRIVE_OUT); OUT.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/xacle"); LOCAL.mkdir(exist_ok=True)
print(DEVICE, torch.__version__)

cuda 2.14.1+cu130


## 3. Tìm dữ liệu trên Drive và chép về ổ local

Notebook tìm thư mục dữ liệu theo ID trên Drive. Đọc 13.500 file wav trực tiếp từ Drive mỗi epoch rất chậm, nên mặc định chép `meta_data/` và `wav/` một lần về `/content/xacle` (khoảng 4 GB, 5–15 phút). Đặt `COPY_LOCAL = False` để đọc thẳng từ Drive.

In [ ]:
MYDRIVE = Path("/content/drive/MyDrive")
BY_ID = Path("/content/drive/.shortcut-targets-by-id")      # nơi Colab mount thư mục được chia sẻ qua lối tắt

def search_drive(pattern):
    hits = sorted(glob.glob(str(MYDRIVE / "**" / pattern), recursive=True), key=len)
    return Path(hits[0]) if hits else None

def locate_dataset():
    cands = [Path(DRIVE_DATA)] if DRIVE_DATA else []
    cands += [BY_ID / DATA_FOLDER_ID,                         # lối tắt tới chính thư mục dữ liệu
              BY_ID / PARENT_FOLDER_ID / DATA_NAME,           # lối tắt tới thư mục cha XACLE
              MYDRIVE / DATA_NAME, MYDRIVE / "XACLE" / DATA_NAME]
    for c in cands:
        if (c / "meta_data" / "train_average.csv").exists():
            return c
    hit = search_drive(f"{DATA_NAME}/meta_data/train_average.csv")   # lối tắt đặt ở thư mục con
    if hit:
        return hit.parent.parent
    raise FileNotFoundError(
        "Không thấy dữ liệu. Hãy thêm lối tắt của thư mục "
        f"https://drive.google.com/drive/folders/{DATA_FOLDER_ID} vào My Drive rồi chạy lại ô này.")

DRIVE_ROOT = locate_dataset()
print("dữ liệu trên Drive:", DRIVE_ROOT)

if COPY_LOCAL:
    DATA_ROOT = LOCAL / DATA_NAME
    if not (DATA_ROOT / "meta_data" / "train_average.csv").exists():
        t0 = time.time()
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["cp", "-r", str(DRIVE_ROOT / "meta_data"), str(DRIVE_ROOT / "wav"), str(DATA_ROOT)], check=True)
        print(f"đã chép về {DATA_ROOT} trong {time.time() - t0:.0f}s")
else:
    DATA_ROOT = DRIVE_ROOT

META_DIR = DATA_ROOT / "meta_data"
WAV_ROOT = DATA_ROOT / "wav"
for sp in ("train", "validation", "test"):
    n = len(list((WAV_ROOT / sp).glob("*.wav")))
    print(f"wav/{sp}: {n} file")

dữ liệu trên Drive: /content/drive/MyDrive/XACLE/XACLE_Challenge_2027_development_dataset
wav/train: 7500 file
wav/validation: 3000 file
wav/test: 3005 file


## 4. Checkpoint M2D-CLAP

Checkpoint chính thức của baseline: `m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth` (bản phát hành v0.5.0 của nttcslab/m2d). Tìm trong Drive trước; không có thì tải từ GitHub.
`PortableM2D` đọc cấu hình từ **tên thư mục cha** của checkpoint, nên file phải nằm trong thư mục đúng tên này.

In [ ]:
CKPT_DIR = Path("/content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = CKPT_DIR / "checkpoint-30.pth"
if not CKPT.exists():
    found = Path(DRIVE_CKPT) if DRIVE_CKPT else search_drive("checkpoint-30.pth")
    if found is not None and found.exists():
        shutil.copy(found, CKPT)
    else:
        url = ("https://github.com/nttcslab/m2d/releases/download/v0.5.0/"
               "m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025.zip")
        subprocess.run(["wget", "-q", "-O", "/content/m2d_ckpt/ckpt.zip", url], check=True)
        with zipfile.ZipFile("/content/m2d_ckpt/ckpt.zip") as z:
            z.extractall("/content/m2d_ckpt")
        if not CKPT.exists():                         # zip có thể lồng thư mục
            CKPT.write_bytes(next(Path("/content/m2d_ckpt").rglob("checkpoint-30.pth")).read_bytes())
print("checkpoint:", CKPT, f"{CKPT.stat().st_size / 1e6:.0f} MB")

checkpoint: /content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth 1695 MB


## 5. Đọc metadata

In [ ]:
def load_split(split):
    df = pd.read_csv(META_DIR / f"{split}_average.csv")
    df.columns = [c.strip() for c in df.columns]           # wav_file_name, text, average_score, model_ID, model_name
    wav_col = next(c for c in df.columns if c.startswith("wav"))                       # wav_file_name / wave_file_name
    y_col = next(c for c in df.columns if "average" in c.lower() and "score" in c.lower())
    if CFG["smoke"]:
        df = df.iloc[:64]
    return dict(
        wavs=[WAV_ROOT / split / str(n).lstrip("/") for n in df[wav_col]],
        texts=[str(t) for t in df["text"]],
        y=df[y_col].to_numpy(np.float32),
        group=df["model_name"].astype(str).str.strip().to_numpy(),
        names=df[wav_col].astype(str).tolist(),
    )

DATA = {s: load_split(s) for s in ("train", "validation", "test")}
for s, d in DATA.items():
    print(s, len(d["y"]), "điểm TB %.2f" % d["y"].mean())
print(pd.Series(DATA["train"]["group"]).value_counts())

train 7500 điểm TB 7.03
validation 3000 điểm TB 6.92
test 3000 điểm TB 7.16
audiocaps         2500
stable_audio       750
audiogen           750
make-an-audio2     750
tangoflux          750
audioldm2          500
tango              500
tango2             500
audioldm           500
Name: count, dtype: int64


## 6. LoRA

`y = W·x + (α/r)·B·A·x`. **W đóng băng**, chỉ học A (r × d_in) và B (d_out × r). B khởi tạo 0 nên ở bước đầu model y hệt M2D-CLAP gốc.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: int):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.zeros(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scale


def _get(m, name):
    return m[int(name)] if name.isdigit() and not hasattr(m, name) else getattr(m, name)

def _set(m, name, v):
    if name.isdigit() and not hasattr(m, name):
        m[int(name)] = v
    else:
        setattr(m, name, v)

def wrap(root, path, r, alpha):
    *parents, last = path.split(".")
    m = root
    for p in parents:
        m = _get(m, p)
    child = _get(m, last)
    assert isinstance(child, nn.Linear), path
    _set(m, last, LoRALinear(child, r, alpha))
    return path

def inject_lora(model):
    done = []
    blocks = model.backbone.blocks
    for i in range(len(blocks) - CFG["lora_blocks"], len(blocks)):
        for sub in ("attn.qkv", "attn.proj"):
            done.append("audio." + wrap(model.backbone, f"blocks.{i}.{sub}", CFG["lora_r"], CFG["lora_alpha"]))
    if CFG["variant"] == "audio_text":
        bert = model.text_encoder.text_encoder
        L = len(bert.encoder.layer)
        for i in range(L - CFG["lora_text_layers"], L):
            for sub in ("attention.self.query", "attention.self.value"):
                done.append("text." + wrap(bert, f"encoder.layer.{i}.{sub}", CFG["lora_r"], CFG["lora_alpha"]))
    return done

## 7. Head, loss CORAL và **thang điểm đã sửa**

- Đặc trưng: instance-norm từng vector, rồi `[a, t, a⊙t, |a−t|, cos]` (3.073 chiều).
- CORAL: điểm làm tròn thành mức 0..10, đầu ra thứ k học xác suất "điểm > k".
- **Điểm dự đoán = Σ sigmoid** (đã nằm sẵn trong [0, 10]). Bản cũ trừ 5 rồi nhân 5 cộng 5 → sai thang.

In [ ]:
class Head(nn.Module):
    def __init__(self, in_dim=4 * 768 + 1, hidden=1024, mid=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, mid), nn.ReLU(), nn.Linear(mid, N_LEVELS))
    def forward(self, x):
        return self.net(x)

def inter_feats(a, t):
    a = (a - a.mean(-1, keepdim=True)) / (a.std(-1, keepdim=True) + 1e-6)
    t = (t - t.mean(-1, keepdim=True)) / (t.std(-1, keepdim=True) + 1e-6)
    return torch.cat([a, t, a * t, (a - t).abs(), (a * t).sum(-1, keepdim=True)], -1).float()

def coral_loss(logits, y):                       # y: điểm gốc 0–10
    lv = torch.clamp(torch.round(y), 0, 10)
    tgt = (lv.unsqueeze(1) > torch.arange(N_LEVELS, device=y.device).float()).float()
    return F.binary_cross_entropy_with_logits(logits.float(), tgt)

def to_score(logits):                            # ĐÃ SỬA: ∈ [0, 10]
    return torch.sigmoid(logits.float()).sum(-1)

## 8. Dữ liệu audio và hàm encode

In [ ]:
def read_wav(path):
    w, _ = sf.read(str(path), dtype="float32", always_2d=True)
    w = w.mean(1)
    n = SR * DUR
    return np.pad(w, (0, max(0, n - len(w))))[:n]

class PairDS(Dataset):
    def __init__(self, d, idx=None):
        self.d, self.idx = d, np.arange(len(d["y"])) if idx is None else np.asarray(idx)
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        return torch.from_numpy(read_wav(self.d["wavs"][i])), self.d["texts"][i], self.d["y"][i]

def collate(b):
    return torch.stack([x[0] for x in b]), [x[1] for x in b], torch.tensor([x[2] for x in b])

def encode_pair(model, wavs, texts):
    x = model.to_normalized_feature(wavs)                     # log-mel fp32
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        frames = model.encode_lms(x)                          # (B, T, 768), ViT + LoRA
    a = model.backbone.audio_proj(frames.float())             # semantic token → 768-d
    t = model.encode_clap_text(texts, truncate=True)          # BERT (+ LoRA nếu B)
    return F.normalize(a.float(), dim=-1), F.normalize(t.float(), dim=-1)

@torch.no_grad()
def predict(model, head, d, idx=None):
    model.eval(); head.eval()
    dl = DataLoader(PairDS(d, idx), batch_size=CFG["eval_batch"], shuffle=False,
                    num_workers=CFG["workers"], collate_fn=collate)
    out = []
    for w, t, _ in dl:
        a, tt = encode_pair(model, w.to(DEVICE, non_blocking=True), t)
        out.append(to_score(head(inter_feats(a, tt))).cpu().numpy())
    return np.concatenate(out)

def metrics(y, p):
    return dict(srcc=float(spearmanr(y, p).statistic), lcc=float(pearsonr(y, p).statistic),
                ktau=float(kendalltau(y, p).statistic), mse=float(np.mean((y - p) ** 2)))

## 9. Vòng train

- 2 nhóm tham số: head (lr 1e-3, có weight decay) và LoRA (lr 5e-4, không weight decay).
- Lịch lr: warmup 10% rồi cosine. Clip gradient 1,0. GradScaler cho fp16.
- `val_split=None` → train đúng `fixed_epochs` epoch, không nhìn nhãn nào (dùng cho LOGO).

In [ ]:
def build_model():
    os.environ.setdefault("HF_HOME", "/tmp/hf_cache")
    m = PortableM2D(str(CKPT), flat_features=True)
    m.get_clap_text_encoder()
    m.requires_grad_(False)
    names = inject_lora(m)
    return m.to(DEVICE), names

def lr_factor(step, total, warm=0.1):
    w = max(1, int(round(warm * total)))
    return (step + 1) / w if step < w else 0.5 * (1 + math.cos(math.pi * min(1.0, (step - w) / max(1, total - w))))

def train_one(train_idx, seed, val_split=None, fixed_epochs=None, tag=""):
    torch.manual_seed(seed); np.random.seed(seed)
    model, lora_names = build_model()
    head = Head().to(DEVICE)
    lora_params = [p for p in model.parameters() if p.requires_grad]
    print(f"{tag} LoRA: {len(lora_names)} lớp, {sum(p.numel() for p in lora_params):,} tham số "
          f"+ head {sum(p.numel() for p in head.parameters()):,}")
    opt = torch.optim.AdamW([
        {"params": head.parameters(), "lr": CFG["lr_head"], "weight_decay": CFG["weight_decay"]},
        {"params": lora_params, "lr": CFG["lr_lora"], "weight_decay": 0.0}])
    base_lr = [CFG["lr_head"], CFG["lr_lora"]]
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    dl = DataLoader(PairDS(DATA["train"], train_idx), batch_size=CFG["batch_size"], shuffle=True,
                    num_workers=CFG["workers"], collate_fn=collate, drop_last=True, persistent_workers=True)
    epochs = 1 if CFG["smoke"] else (fixed_epochs or CFG["epochs"])
    total, step = len(dl) * epochs, 0
    best, best_ep, best_state, bad, hist = -9.0, 0, None, 0, []
    for ep in range(1, epochs + 1):
        model.train(); head.train(); t0 = time.time()
        for w, t, y in dl:
            a, tt = encode_pair(model, w.to(DEVICE, non_blocking=True), t)
            loss = coral_loss(head(inter_feats(a, tt)), y.to(DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([*head.parameters(), *lora_params], 1.0)
            scaler.step(opt); scaler.update()
            step += 1
            for g, lr in zip(opt.param_groups, base_lr):
                g["lr"] = lr * lr_factor(step, total)
        if val_split is None:                       # LOGO: không early stop
            print(f"{tag} ep{ep} ({time.time() - t0:.0f}s)")
            best_ep = ep
            continue
        pv = predict(model, head, val_split)
        s = metrics(val_split["y"], pv)["srcc"]; hist.append(s)
        print(f"{tag} ep{ep}: val SRCC {s:.4f} ({time.time() - t0:.0f}s)")
        if s > best + 1e-5:
            best, best_ep, bad = s, ep, 0
            best_state = ({k: v.detach().clone() for k, v in head.state_dict().items()},
                          [p.detach().clone() for p in lora_params])
        else:
            bad += 1
            if bad >= CFG["patience"]:
                print(f"{tag} early stop"); break
    if best_state is not None:
        head.load_state_dict(best_state[0])
        for p, v in zip(lora_params, best_state[1]):
            p.data.copy_(v)
    return model, head, lora_params, best_ep, hist

## 10. Lượt chính: train trên split `train`, chọn epoch theo validation, đo val và test

In [ ]:
t0 = time.time()
model, head, lora_params, best_ep, hist = train_one(None, CFG["seed"], val_split=DATA["validation"], tag="main")
pv = predict(model, head, DATA["validation"])
pt = predict(model, head, DATA["test"])
res = {"val": metrics(DATA["validation"]["y"], pv), "test": metrics(DATA["test"]["y"], pt),
       "best_epoch": best_ep, "val_curve": hist, "minutes": round((time.time() - t0) / 60, 1), "cfg": CFG}
print(json.dumps({k: res[k] for k in ("val", "test", "best_epoch")}, indent=2))

# lưu dự đoán của ĐÚNG model này và trọng số để tái lập
for s, p in (("validation", pv), ("test", pt)):
    pd.DataFrame({"wav_file_name": DATA[s]["names"], "prediction": p}).to_csv(OUT / f"pred_{s}_seed{CFG['seed']}.csv", index=False)
torch.save({"head": head.state_dict(), "lora": [p.cpu() for p in lora_params], "cfg": CFG}, OUT / f"lora_{CFG['variant']}_seed{CFG['seed']}.pt")
(OUT / "metrics_main.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

 using 166 parameters from /content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth
 (included audio_proj params: ['audio_proj.sem_token', 'audio_proj.sem_blocks.0.norm1.weight', 'audio_proj.sem_blocks.0.norm1.bias', 'audio_proj.sem_blocks.0.attn.qkv.weight', 'audio_proj.sem_blocks.0.attn.qkv.bias']
 (included text_proj params: []
 (dropped: [] )
<All keys matched successfully>


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 using model.text_encoder from /content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth
main LoRA: 12 lớp, 196,608 tham số + head 3,412,746
main ep1: val SRCC 0.6210 (107s)
main ep2: val SRCC 0.6304 (106s)
main ep3: val SRCC 0.6515 (106s)
main ep4: val SRCC 0.6478 (106s)
main ep5: val SRCC 0.6487 (106s)
main early stop
{
  "val": {
    "srcc": 0.6514548433477185,
    "lcc": 0.6707679033279419,
    "ktau": 0.47320573963445606,
    "mse": 3.1471920013427734
  },
  "test": {
    "srcc": 0.645711513769665,
    "lcc": 0.7003104090690613,
    "ktau": 0.46503436007951887,
    "mse": 2.63777494430542
  },
  "best_epoch": 3
}


821

## 11. (Tuỳ chọn) LOGO 9 fold — đo khả năng tổng quát hoá sang generator chưa thấy

Mỗi fold bỏ trọn một generator khỏi train, train **đúng `best_epoch` epoch** (lấy từ lượt chính, không early-stop trên fold), rồi đo SRCC trên generator bị bỏ. So với head trên embedding đóng băng: **LOGO macro 0,570** (inst-inter-coral).

In [ ]:
if CFG["run_logo"]:
    del model, head; torch.cuda.empty_cache()
    g = DATA["train"]["group"]
    fold = {}
    for gen in sorted(set(g)):
        held, keep = np.where(g == gen)[0], np.where(g != gen)[0]
        m, h, _, _, _ = train_one(keep, CFG["seed"], fixed_epochs=max(1, best_ep), tag=f"[{gen}]")
        p = predict(m, h, DATA["train"], held)
        fold[gen] = metrics(DATA["train"]["y"][held], p)["srcc"]
        print(f"LOGO {gen}: {fold[gen]:.4f}")
        del m, h; torch.cuda.empty_cache()
    res["logo_folds"] = fold
    res["logo_macro"] = float(np.mean(list(fold.values())))
    print("LOGO macro:", round(res["logo_macro"], 4), "| mốc head đóng băng: 0.570")
    (OUT / "metrics_main.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

## Ghi chú

- Chỉ dùng split `train` để học; validation chỉ để chọn epoch (giống baseline). Không gộp nhiều seed.
- Muốn so A với B: đổi `CFG["variant"]` và chạy lại.
- LOGO 9 fold với `best_epoch` ≈ 3 tốn khoảng 9 × 3 × ~100 s ≈ 45–60 phút trên 1 T4.